In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
cleaned_df = pd.read_csv('cleaned_messages.csv')
cleaned_messages = cleaned_df['message'].tolist()

print(f"Total cleaned messages: {len(cleaned_messages)}")
print(f"\nFirst 5 cleaned messages:")
for i, msg in enumerate(cleaned_messages[:5], 1):
    print(f"{i}. {msg}")

Total cleaned messages: 1999

First 5 cleaned messages:
1. packetresponder <num> for block <block_id> terminating
2. packetresponder <num> for block <block_id> terminating
3. block* namesystem.addstoredblock: blockmap updated: <ip>:<port> is added to <block_id> size <large_num>
4. packetresponder <num> for block <block_id> terminating
5. packetresponder <num> for block <block_id> terminating


In [3]:
def extract_tfidf_features(messages, max_features=50): #keep top 50 most informative tokens
    tfidf_vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        lowercase=False
    )
    
    tfidf_matrix = tfidf_vectorizer.fit_transform(messages)
    tfidf_features = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=[f'tfidf_{i}' for i in range(tfidf_matrix.shape[1])]
    )
    print("TF-IDF shape:", tfidf_matrix.shape)
    print("Top 10 features:", tfidf_vectorizer.get_feature_names_out()[:10])

    return tfidf_matrix,tfidf_features, tfidf_vectorizer


In [4]:
tfidf_matrix,tfidf_features, tfidf_vectorizer = extract_tfidf_features(cleaned_messages, max_features=50)

TF-IDF shape: (1999, 50)
Top 10 features: ['added' 'added to' 'addstoredblock' 'addstoredblock blockmap' 'block'
 'block block_id' 'block namesystem' 'block_id of' 'block_id size'
 'block_id src']


In [5]:
tfidf_vectorizer

TfidfVectorizer(lowercase=False, max_df=0.95, max_features=50, min_df=2,
                ngram_range=(1, 2))

In [6]:
from scipy.sparse import csr_matrix
dense_matrix = tfidf_matrix.toarray()

In [7]:
dense_matrix

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.19607691, 0.19607691, 0.24164715, ..., 0.24164715, 0.24164715,
        0.24164715],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [8]:
tfidf_features

,tfidf_0,tfidf_1,tfidf_2,tfidf_3,tfidf_4,tfidf_5,tfidf_6,tfidf_7,tfidf_8,tfidf_9,...,tfidf_40,tfidf_41,tfidf_42,tfidf_43,tfidf_44,tfidf_45,tfidf_46,tfidf_47,tfidf_48,tfidf_49
0,0.000000,0.000000,0.000000,0.000000,0.128839,0.181038,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.350243,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.128839,0.181038,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.350243,0.000000,0.000000,0.000000,0.000000
2,0.196077,0.196077,0.241647,0.241647,0.089190,0.000000,0.17967,0.000000,0.241647,0.000000,...,0.000000,0.185857,0.185857,0.000000,0.000000,0.000000,0.173179,0.241647,0.241647,0.241647
3,0.000000,0.000000,0.000000,0.000000,0.128839,0.181038,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.350243,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.128839,0.181038,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.350243,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1994,0.000000,0.000000,0.000000,0.000000,0.103841,0.145913,0.00000,0.000000,0.000000,0.287822,...,0.288494,0.000000,0.000000,0.287822,0.287822,0.000000,0.000000,0.000000,0.000000,0.000000
1995,0.000000,0.000000,0.000000,0.000000,0.114006,0.160196,0.00000,0.317477,0.000000,0.000000,...,0.000000,0.237570,0.237570,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1996,0.000000,0.000000,0.000000,0.000000,0.103841,0.145913,0.00000,0.000000,0.000000,0.287822,...,0.288494,0.000000,0.000000,0.287822,0.287822,0.000000,0.000000,0.000000,0.000000,0.000000
1997,0.000000,0.000000,0.000000,0.000000,0.128839,0.181038,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.350243,0.000000,0.000000,0.000000,0.000000


In [9]:
tfidf_features.to_csv('tfidf_features.csv',index=False)

In [10]:
import joblib

joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']